# ShopEase E-Commerce Sales Analysis
### Celebal Summer Internship 2026 — Week 2 SQL Task
**Analyst:** Junior Data Analyst | **Engine:** SQLite (standard SQL)

**Tables:** `customers` · `products` · `orders` · `order_items`

---

## Part 0 — Database Setup

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")
cur = con.cursor()

cur.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name  TEXT NOT NULL,
    last_name   TEXT NOT NULL,
    email       TEXT UNIQUE NOT NULL,
    city        TEXT NOT NULL,
    state       TEXT NOT NULL,
    join_date   TEXT NOT NULL,
    is_premium  INTEGER DEFAULT 0
);
CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category     TEXT NOT NULL,
    brand        TEXT NOT NULL,
    unit_price   REAL NOT NULL CHECK (unit_price > 0),
    stock_qty    INTEGER NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
);
CREATE TABLE orders (
    order_id     INTEGER PRIMARY KEY,
    customer_id  INTEGER NOT NULL,
    order_date   TEXT NOT NULL,
    status       TEXT NOT NULL DEFAULT 'Pending'
                 CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount REAL NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
CREATE TABLE order_items (
    item_id      INTEGER PRIMARY KEY,
    order_id     INTEGER NOT NULL,
    product_id   INTEGER NOT NULL,
    quantity     INTEGER NOT NULL CHECK (quantity > 0),
    unit_price   REAL NOT NULL CHECK (unit_price > 0),
    discount_pct REAL DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    FOREIGN KEY (order_id)   REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")
print("Schema created.")

In [ ]:
cur.executescript("""
INSERT INTO customers VALUES
(101,'Aarav','Sharma','aarav.s@email.com','Mumbai','Maharashtra','2024-01-15',1),
(102,'Priya','Patel','priya.p@email.com','Ahmedabad','Gujarat','2024-02-20',0),
(103,'Rohan','Gupta','rohan.g@email.com','Delhi','Delhi','2024-03-10',1),
(104,'Sneha','Reddy','sneha.r@email.com','Hyderabad','Telangana','2024-04-05',0),
(105,'Vikram','Singh','vikram.s@email.com','Jaipur','Rajasthan','2024-05-12',1),
(106,'Ananya','Iyer','ananya.i@email.com','Chennai','Tamil Nadu','2024-06-18',0),
(107,'Karan','Mehta','karan.m@email.com','Pune','Maharashtra','2024-07-22',1),
(108,'Divya','Nair','divya.n@email.com','Kochi','Kerala','2024-08-30',0);
INSERT INTO products VALUES
(201,'Wireless Earbuds','Electronics','BoAt',1499.00,250),
(202,'Cotton T-Shirt','Clothing','Levis',799.00,500),
(203,'Smart Watch','Electronics','Noise',2999.00,150),
(204,'Running Shoes','Clothing','Nike',4599.00,120),
(205,'Bluetooth Speaker','Electronics','JBL',3499.00,200),
(206,'Bedsheet Set','Home','Spaces',1299.00,300),
(207,'Laptop Stand','Electronics','AmazonBasics',899.00,180),
(208,'Cushion Covers (Set)','Home','HomeCenter',599.00,400);
INSERT INTO orders VALUES
(1001,101,'2024-08-01','Delivered',4498.00),
(1002,102,'2024-08-03','Delivered',799.00),
(1003,103,'2024-08-05','Shipped',7498.00),
(1004,101,'2024-08-10','Delivered',3499.00),
(1005,104,'2024-08-12','Cancelled',2999.00),
(1006,105,'2024-08-15','Delivered',5898.00),
(1007,106,'2024-08-18','Pending',1299.00),
(1008,103,'2024-08-20','Delivered',899.00),
(1009,107,'2024-08-25','Shipped',6098.00),
(1010,108,'2024-08-28','Delivered',1598.00);
INSERT INTO order_items VALUES
(5001,1001,201,2,1499.00,0),(5002,1001,207,1,899.00,10),
(5003,1002,202,1,799.00,0),(5004,1003,203,1,2999.00,0),
(5005,1003,204,1,4599.00,5),(5006,1004,205,1,3499.00,0),
(5007,1005,203,1,2999.00,0),(5008,1006,201,1,1499.00,10),
(5009,1006,204,1,4599.00,5),(5010,1007,206,1,1299.00,0),
(5011,1008,207,1,899.00,0),(5012,1009,205,1,3499.00,0),
(5013,1009,208,2,599.00,15),(5014,1010,206,1,1299.00,0),
(5015,1010,208,1,599.00,0);
""")

def run(sql):
    return pd.read_sql_query(sql, con)

print("Data loaded. Tables ready.")

---
## Section A — SQL Basics

### Q1 — All rows from customers

In [ ]:
run('SELECT * FROM customers')

### Q2 — first_name, last_name, city

In [ ]:
run('SELECT first_name, last_name, city FROM customers')

### Q3 — Unique product categories

In [ ]:
run('SELECT DISTINCT category FROM products')

### Q4 — Primary Keys

| Table | Primary Key |
|---|---|
| customers | customer_id |
| products | product_id |
| orders | order_id |
| order_items | item_id |

**Why UNIQUE + NOT NULL?** A Primary Key must uniquely identify every row. UNIQUE prevents two rows sharing the same ID; NOT NULL ensures the identifier always exists — a NULL ID cannot be referenced by foreign keys.

### Q5 — Constraints on `email`

- **UNIQUE** — no two customers share the same email
- **NOT NULL** — every customer must have an email

On a duplicate insert: `ERROR 1062` (MySQL) / `ERROR 23505` (PostgreSQL) — the INSERT is rejected entirely.

### Q6 — Inserting `unit_price = -50`

```sql
INSERT INTO products (..., unit_price, ...) VALUES (..., -50.00, ...);
```
Blocked by `CHECK (unit_price > 0)` — raises a check-constraint violation error.

In [ ]:
try:
    con.execute("""INSERT INTO products
        (product_id, product_name, category, brand, unit_price, stock_qty)
        VALUES (299,'Test','Electronics','Brand',-50.00,10)""")
except Exception as e:
    print(f"Constraint violation: {e}")

---
## Section B — Filtering & Optimization

### Q7 — All Delivered orders

In [ ]:
run("SELECT * FROM orders WHERE status = 'Delivered'")
# 6 of 10 orders (60%) delivered

### Q8 — Electronics products priced above ₹2000

In [ ]:
run("""SELECT * FROM products
WHERE category = 'Electronics' AND unit_price > 2000""")

### Q9 — Maharashtra customers who joined in 2024

In [ ]:
# SARGable range — keeps the index usable
run("""SELECT * FROM customers
WHERE join_date >= '2024-01-01'
  AND join_date <  '2025-01-01'
  AND state = 'Maharashtra'""")

### Q10 — Orders Aug 10–25, excluding Cancelled

In [ ]:
run("""SELECT * FROM orders
WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25'
  AND status <> 'Cancelled'""")

### Q11 — Index `idx_orders_date` and performance

`idx_orders_date` is a **B-Tree index** on `orders(order_date)`.

| | Without Index | With Index |
|---|---|---|
| Method | Full table scan | B-Tree range seek |
| Complexity | O(n) | O(log n) |

The date-range query below uses this index for an efficient seek instead of reading every row.

In [ ]:
run("""SELECT * FROM orders
WHERE order_date BETWEEN '2024-08-01' AND '2024-08-31'""")

### Q12 — SARGable rewrite for year filter

**Non-SARGable (index bypassed):**
```sql
SELECT * FROM customers WHERE YEAR(join_date) = 2024;
```
Wrapping the column in `YEAR()` forces a full scan.

**SARGable (index used):**

In [ ]:
run("""SELECT * FROM customers
WHERE join_date >= '2024-01-01'
  AND join_date <  '2025-01-01'""")

---
## Section C — Aggregation

### Q13 — Total number of orders

In [ ]:
run('SELECT COUNT(*) AS total_orders FROM orders')

### Q14 — Total revenue from Delivered orders

In [ ]:
run("""SELECT SUM(total_amount) AS total_revenue
FROM orders WHERE status = 'Delivered'""")
# Result: 17191.00 (~48% of gross 35085)

### Q15 — Average unit_price per category

In [ ]:
run("""SELECT category, ROUND(AVG(unit_price),2) AS avg_price
FROM products GROUP BY category""")

### Q16 — Order count & revenue by status

In [ ]:
run("""SELECT status,
       COUNT(*) AS order_count,
       ROUND(SUM(total_amount),2) AS total_revenue
FROM orders
GROUP BY status
ORDER BY total_revenue DESC""")

### Q17 — Max and Min price per category

In [ ]:
run("""SELECT category,
       MAX(unit_price) AS max_price,
       MIN(unit_price) AS min_price
FROM products GROUP BY category""")

### Q18 — Categories where avg price > ₹2000 (HAVING)

In [ ]:
run("""SELECT category, ROUND(AVG(unit_price),2) AS avg_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2000""")
# Home (avg 949) excluded; only Clothing & Electronics qualify

---
## Section D — Joins & Relationships

### Q19 — Orders with customer name (INNER JOIN)

In [ ]:
run("""SELECT o.order_id, o.order_date,
       c.first_name, c.last_name, o.total_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id""")

### Q20 — All customers + their orders (LEFT JOIN)

In [ ]:
run("""SELECT c.customer_id, c.first_name, c.last_name,
       o.order_id, o.total_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id""")

### Q21 — Order items with product details (3-table JOIN)

In [ ]:
run("""SELECT oi.order_id, p.product_name,
       oi.quantity, oi.unit_price, oi.discount_pct
FROM order_items oi
JOIN orders   o ON oi.order_id   = o.order_id
JOIN products p ON oi.product_id = p.product_id""")

### Q22 — LEFT vs RIGHT vs FULL OUTER JOIN

| Join | Returns |
|---|---|
| **LEFT JOIN** | All left rows + matched right (unmatched right → NULL) |
| **RIGHT JOIN** | All right rows + matched left (unmatched left → NULL) |
| **FULL OUTER JOIN** | All rows from both sides (NULLs where no match) |

> MySQL does not support FULL OUTER JOIN natively — emulate with `LEFT JOIN UNION RIGHT JOIN`.

### Q23 — Foreign Key Relationships

```
customers.customer_id ◀── orders.customer_id
orders.order_id       ◀── order_items.order_id
products.product_id   ◀── order_items.product_id
```
Inserting `customer_id = 999` (non-existent) raises `ERROR 1452` (MySQL) — referential integrity enforced.

In [ ]:
try:
    con.execute("""INSERT INTO orders
        (order_id, customer_id, order_date, status, total_amount)
        VALUES (1099, 999, '2024-09-01', 'Pending', 500.00)""")
except Exception as e:
    print(f"FK violation: {e}")

---
## Section E — Advanced Concepts

### Q24 — Price tiers with CASE

In [ ]:
run("""SELECT product_name, unit_price,
       CASE
           WHEN unit_price < 1000 THEN 'Budget'
           WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
           ELSE 'Premium'
       END AS price_tier
FROM products""")
# 3 Budget | 3 Mid-Range | 2 Premium

### Q25 — Delivered vs Not-Delivered in a single row

In [ ]:
run("""SELECT
    SUM(CASE WHEN status = 'Delivered'  THEN 1 ELSE 0 END) AS delivered,
    SUM(CASE WHEN status <> 'Delivered' THEN 1 ELSE 0 END) AS not_delivered
FROM orders""")
# 60% fulfilment rate

### Q26 — ACID Properties

| | Property | Description |
|---|---|---|
| **A** | Atomicity | All steps succeed or all are rolled back |
| **C** | Consistency | DB moves between valid states only |
| **I** | Isolation | Concurrent transactions don't interfere |
| **D** | Durability | Committed data survives crashes |

**Example:** A bank transfer of ₹5,000 — if the credit step fails, the debit is automatically undone.

### Q27 — Full Transaction: New order + 2 items + stock update

In [ ]:
try:
    con.execute("BEGIN")
    con.execute("""INSERT INTO orders
        (order_id, customer_id, order_date, status, total_amount)
        VALUES (1011, 102, date('now'), 'Pending', 1598.00)""")
    con.execute("""INSERT INTO order_items
        (item_id, order_id, product_id, quantity, unit_price, discount_pct)
        VALUES (5016, 1011, 206, 1, 1299.00, 0)""")
    con.execute("""INSERT INTO order_items
        (item_id, order_id, product_id, quantity, unit_price, discount_pct)
        VALUES (5017, 1011, 208, 1, 599.00, 0)""")
    con.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206")
    con.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208")
    con.execute("COMMIT")
    print("Transaction committed successfully.")
except Exception as e:
    con.execute("ROLLBACK")
    print(f"Rolled back: {e}")

In [ ]:
print("New order:")
display(run("SELECT * FROM orders WHERE order_id = 1011"))
print("Updated stock:")
display(run("SELECT product_id, product_name, stock_qty FROM products WHERE product_id IN (206,208)"))

---
## Validation — Data Quality Checks

In [ ]:
checks = {
    "Row counts": """
        SELECT 'customers' AS tbl, COUNT(*) AS rows FROM customers UNION ALL
        SELECT 'products',         COUNT(*)           FROM products  UNION ALL
        SELECT 'orders',           COUNT(*)           FROM orders    UNION ALL
        SELECT 'order_items',      COUNT(*)           FROM order_items""",
    "Null emails":
        "SELECT COUNT(*) AS null_emails FROM customers WHERE email IS NULL",
    "Duplicate emails":
        "SELECT email, COUNT(*) AS cnt FROM customers GROUP BY email HAVING COUNT(*) > 1",
    "Orphaned orders":
        """SELECT o.order_id FROM orders o
        LEFT JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL""",
    "Negative stock":
        "SELECT product_id, product_name, stock_qty FROM products WHERE stock_qty < 0",
}
for label, sql in checks.items():
    df = run(sql)
    print(f"--- {label} ---")
    print(df.to_string(index=False))
    print()

---
## Bonus — Business Insights

### Top 3 customers by spend (Delivered orders)

In [ ]:
run("""SELECT c.customer_id,
       c.first_name || ' ' || c.last_name AS customer_name,
       COUNT(o.order_id)  AS total_orders,
       SUM(o.total_amount) AS total_spend
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'Delivered'
GROUP BY c.customer_id
ORDER BY total_spend DESC
LIMIT 3""")

### Best-selling products by quantity

In [ ]:
run("""SELECT p.product_name, SUM(oi.quantity) AS total_qty_sold
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_qty_sold DESC
LIMIT 5""")

### Net revenue contribution by category (after discounts)

In [ ]:
run("""SELECT p.category,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct/100)), 2) AS net_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY net_revenue DESC""")
# Electronics: 22574.70 (~64% of total)

---
## Key Insights Summary

| # | Insight |
|---|---|
| 1 | 60% fulfilment rate — 6/10 orders Delivered; ₹13,596 still in-transit |
| 2 | Top customer: Aarav Sharma — ₹7,997 across 2 delivered orders |
| 3 | Electronics dominates with ~64% of net revenue (₹22,575) |
| 4 | Both Maharashtra customers are Premium members |
| 5 | Wireless Earbuds and Cushion Covers are top sellers by quantity |
| 6 | Home is budget-focused (avg ₹949); good for volume |
| 7 | Zero data quality issues — no NULLs, duplicates, or orphaned records |